Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 1
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC query = """
# MAGIC
# MAGIC WITH nc_tickets AS (
# MAGIC     SELECT
# MAGIC         canonical_ticket_id,
# MAGIC         customer_id,
# MAGIC         CAST(first_ticket_created_date AS DATE) AS ticket_dt,
# MAGIC         CAST(first_call_timestamp AS TIMESTAMP) AS first_call_ts,
# MAGIC         total_calls,
# MAGIC         agents_involved,
# MAGIC         sticky_agent_name,
# MAGIC         agents_sequential,
# MAGIC         call_creation_timestamps
# MAGIC     FROM corp_analytics.pl_apex_canonical_tickets_tbl
# MAGIC     WHERE disposition_reason in ('Short Call','Not Connected')
# MAGIC       AND DATE(first_ticket_created_date)
# MAGIC             BETWEEN CURRENT_DATE - INTERVAL '15' DAY AND CURRENT_DATE
# MAGIC       AND total_calls < 10
# MAGIC       AND chat_ticket_state NOT IN ('FORCE_CLOSED', 'RESOLVED')
# MAGIC ),
# MAGIC
# MAGIC cust_bsd AS (
# MAGIC     SELECT
# MAGIC         customer_id,
# MAGIC         bsd_tier
# MAGIC     FROM (
# MAGIC         SELECT
# MAGIC             o.customer_reference_id AS customer_id,
# MAGIC             TRY_CAST(s.bsdv1_retrain_tier AS INTEGER) AS bsd_tier,
# MAGIC
# MAGIC             ROW_NUMBER() OVER (
# MAGIC                 PARTITION BY o.customer_reference_id
# MAGIC                 ORDER BY o.loan_offer_reference_id DESC
# MAGIC             ) AS rn
# MAGIC
# MAGIC         FROM corp_analytics.customer_last_approved_offer_tbl o
# MAGIC
# MAGIC         INNER JOIN corp_analytics.pl_uw_scores_tiers_tbl s
# MAGIC             ON s.loan_offer_reference_id = o.loan_offer_reference_id
# MAGIC
# MAGIC         WHERE TRY_CAST(s.bsdv1_retrain_tier AS INTEGER)
# MAGIC               BETWEEN 1 AND 10
# MAGIC     ) ranked
# MAGIC     WHERE rn = 1
# MAGIC ),
# MAGIC
# MAGIC latest_active_stock AS (
# MAGIC     SELECT
# MAGIC         t.canonical_ticket_id,
# MAGIC         s.customer_reference_id,
# MAGIC         s.offer_max_loan_amount,
# MAGIC         s.expiry_date,
# MAGIC
# MAGIC         DATE(SUBSTRING(s.stock_date, 1, 10)) AS stock_dt,
# MAGIC
# MAGIC         ROW_NUMBER() OVER (
# MAGIC             PARTITION BY t.canonical_ticket_id
# MAGIC             ORDER BY DATE(SUBSTRING(s.stock_date, 1, 10)) DESC
# MAGIC         ) AS rn
# MAGIC
# MAGIC     FROM nc_tickets t
# MAGIC
# MAGIC     INNER JOIN corp_analytics.pl_stock_v1_tbl s
# MAGIC         ON t.customer_id = s.customer_reference_id
# MAGIC        AND DATE(SUBSTRING(s.stock_date, 1, 10)) <= t.ticket_dt
# MAGIC        AND DATE(s.expiry_date) >= t.ticket_dt
# MAGIC        AND DATE(SUBSTRING(s.stock_date, 1, 10))
# MAGIC             >= CURRENT_DATE - INTERVAL '6' MONTH
# MAGIC ),
# MAGIC
# MAGIC o2d_stages AS (
# MAGIC     SELECT
# MAGIC         customer_id,
# MAGIC         DATE(CAST(event_date AS timestamp)) AS event_dt,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'OFFER_NEXT'
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS offer_next_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'PENNY'
# MAGIC                      AND COALESCE(is_silent_penny, 0) = 0
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS penny_init_actual_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'PENNY'
# MAGIC                      AND status = 'SUCCESS'
# MAGIC                      AND COALESCE(is_silent_penny, 0) = 0
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS penny_success_actual_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'MANDATE'
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS mandate_init_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'MANDATE'
# MAGIC                      AND status IN ('SUCCESS', 'ACTIVE', 'REGISTERED')
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS mandate_success_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'KYC'
# MAGIC                      AND COALESCE(is_kyc_replication, 0) = 0
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS kyc_success_actual_ts,
# MAGIC
# MAGIC         MIN(
# MAGIC             CASE
# MAGIC                 WHEN stage_tag = 'LOAN_SUMMARY_NEXT'
# MAGIC                 THEN created_ts_ist
# MAGIC             END
# MAGIC         ) AS loan_summary_next_ts
# MAGIC
# MAGIC     FROM corp_analytics.pl_raw_stage_tbl
# MAGIC     WHERE CAST(event_date AS timestamp) >= TIMESTAMP '2026-02-01 00:00:00'
# MAGIC       AND stage_tag IN (
# MAGIC             'OFFER_NEXT',
# MAGIC             'PENNY',
# MAGIC             'MANDATE',
# MAGIC             'KYC',
# MAGIC             'LOAN_SUMMARY_NEXT'
# MAGIC       )
# MAGIC       AND customer_id IN (SELECT customer_id FROM nc_tickets)
# MAGIC
# MAGIC     GROUP BY 1, 2
# MAGIC ),
# MAGIC
# MAGIC o2d_post_ticket AS (
# MAGIC     SELECT
# MAGIC         a.canonical_ticket_id,
# MAGIC         s.offer_next_ts,
# MAGIC         s.penny_init_actual_ts,
# MAGIC         s.penny_success_actual_ts,
# MAGIC         s.mandate_init_ts,
# MAGIC         s.mandate_success_ts,
# MAGIC         s.kyc_success_actual_ts,
# MAGIC         s.loan_summary_next_ts
# MAGIC     FROM nc_tickets a
# MAGIC     JOIN o2d_stages s
# MAGIC         ON a.customer_id = s.customer_id
# MAGIC        AND CAST(s.event_dt AS date)
# MAGIC             >= CAST(a.ticket_dt AS date)
# MAGIC        AND CAST(s.event_dt AS date)
# MAGIC             <= CAST(a.ticket_dt AS date) + INTERVAL '7' DAY
# MAGIC ),
# MAGIC
# MAGIC o2d_consolidated AS (
# MAGIC     SELECT
# MAGIC         canonical_ticket_id,
# MAGIC         MIN(offer_next_ts) AS offer_next_ts,
# MAGIC         MIN(penny_init_actual_ts) AS penny_init_actual_ts,
# MAGIC         MIN(penny_success_actual_ts) AS penny_success_actual_ts,
# MAGIC         MIN(mandate_init_ts) AS mandate_init_ts,
# MAGIC         MIN(mandate_success_ts) AS mandate_success_ts,
# MAGIC         MIN(kyc_success_actual_ts) AS kyc_success_actual_ts,
# MAGIC         MIN(loan_summary_next_ts) AS loan_summary_next_ts
# MAGIC     FROM o2d_post_ticket
# MAGIC     GROUP BY 1
# MAGIC ),
# MAGIC
# MAGIC sales AS (
# MAGIC     SELECT
# MAGIC         customer_id,
# MAGIC         CAST(disbursed_date AS date) AS disbursal_date
# MAGIC     FROM corp_analytics.pl_sales_fct_tbl
# MAGIC     WHERE CAST(disbursed_date AS date) >= DATE '2026-02-01'
# MAGIC       AND customer_id IN (SELECT customer_id FROM nc_tickets)
# MAGIC ),
# MAGIC
# MAGIC disbursal_match AS (
# MAGIC     SELECT
# MAGIC         a.canonical_ticket_id,
# MAGIC         MIN(s.disbursal_date) AS funnel_disbursal_date
# MAGIC     FROM nc_tickets a
# MAGIC     JOIN sales s
# MAGIC         ON a.customer_id = s.customer_id
# MAGIC        AND CAST(s.disbursal_date AS date)
# MAGIC             >= CAST(a.ticket_dt AS date)
# MAGIC        AND CAST(s.disbursal_date AS date)
# MAGIC             <= CAST(a.ticket_dt AS date) + INTERVAL '7' DAY
# MAGIC     GROUP BY 1
# MAGIC ),
# MAGIC
# MAGIC eligible_tickets AS (
# MAGIC     SELECT
# MAGIC         t.*,
# MAGIC         st.offer_max_loan_amount,
# MAGIC         COALESCE(cb.bsd_tier, -1) AS bsd_tier,
# MAGIC         CASE
# MAGIC             WHEN d.funnel_disbursal_date IS NOT NULL
# MAGIC                 THEN 'DISBURSED'
# MAGIC             WHEN o.loan_summary_next_ts IS NOT NULL
# MAGIC                 THEN 'LOAN_SUMMARY'
# MAGIC             WHEN o.kyc_success_actual_ts IS NOT NULL
# MAGIC                 THEN 'KYC_SUCCESS'
# MAGIC             WHEN o.mandate_success_ts IS NOT NULL
# MAGIC                 THEN 'MANDATE_SUCCESS'
# MAGIC             WHEN o.mandate_init_ts IS NOT NULL
# MAGIC                 THEN 'MANDATE_INIT'
# MAGIC             WHEN o.penny_success_actual_ts IS NOT NULL
# MAGIC                 THEN 'PENNY_SUCCESS'
# MAGIC             WHEN o.penny_init_actual_ts IS NOT NULL
# MAGIC                 THEN 'PENNY_INIT'
# MAGIC             WHEN o.offer_next_ts IS NOT NULL
# MAGIC                 THEN 'OFFER_NEXT'
# MAGIC             ELSE 'OFFER_ONLY'
# MAGIC         END AS furthest_stage
# MAGIC
# MAGIC     FROM nc_tickets t
# MAGIC
# MAGIC     INNER JOIN latest_active_stock st
# MAGIC         ON t.canonical_ticket_id = st.canonical_ticket_id
# MAGIC        AND st.rn = 1
# MAGIC
# MAGIC     LEFT JOIN cust_bsd cb
# MAGIC         ON t.customer_id = cb.customer_id
# MAGIC
# MAGIC     LEFT JOIN o2d_consolidated o
# MAGIC         ON o.canonical_ticket_id = t.canonical_ticket_id
# MAGIC
# MAGIC     LEFT JOIN disbursal_match d
# MAGIC         ON d.canonical_ticket_id = t.canonical_ticket_id
# MAGIC ),
# MAGIC
# MAGIC call_level AS (
# MAGIC     SELECT
# MAGIC         t.canonical_ticket_id,
# MAGIC         t.customer_id,
# MAGIC         t.ticket_dt,
# MAGIC         t.first_call_ts,
# MAGIC         t.total_calls,
# MAGIC         t.agents_involved,
# MAGIC         t.sticky_agent_name,
# MAGIC         t.agents_sequential,
# MAGIC         t.offer_max_loan_amount,
# MAGIC         t.bsd_tier,
# MAGIC         t.furthest_stage,
# MAGIC         t.call_creation_timestamps,
# MAGIC
# MAGIC         CAST(call_ts AS TIMESTAMP) AS call_ts
# MAGIC
# MAGIC     FROM eligible_tickets t
# MAGIC
# MAGIC     LEFT JOIN UNNEST(
# MAGIC         CAST(JSON_PARSE(t.call_creation_timestamps) AS ARRAY(VARCHAR))
# MAGIC     ) AS u(call_ts)
# MAGIC     ON TRUE
# MAGIC )
# MAGIC
# MAGIC SELECT
# MAGIC     customer_id,
# MAGIC     canonical_ticket_id,
# MAGIC     ticket_dt,
# MAGIC
# MAGIC     first_call_ts AS first_call_date_timestamp,
# MAGIC
# MAGIC     total_calls AS total_calls_made_on_ticket,
# MAGIC
# MAGIC     COUNT(
# MAGIC         CASE
# MAGIC             WHEN DATE(call_ts) = ticket_dt
# MAGIC             THEN 1
# MAGIC         END
# MAGIC     ) AS calls_day_0,
# MAGIC
# MAGIC     COUNT(
# MAGIC         CASE
# MAGIC             WHEN DATE(call_ts) = ticket_dt + INTERVAL '1' DAY
# MAGIC             THEN 1
# MAGIC         END
# MAGIC     ) AS calls_day_1,
# MAGIC
# MAGIC     COUNT(
# MAGIC         CASE
# MAGIC             WHEN DATE(call_ts) = ticket_dt + INTERVAL '2' DAY
# MAGIC             THEN 1
# MAGIC         END
# MAGIC     ) AS calls_day_2,
# MAGIC
# MAGIC     agents_involved AS agent_name,
# MAGIC
# MAGIC     bsd_tier,
# MAGIC
# MAGIC     offer_max_loan_amount,
# MAGIC
# MAGIC     furthest_stage,
# MAGIC
# MAGIC     MAX(call_creation_timestamps) AS call_creation_timestamps,
# MAGIC
# MAGIC     sticky_agent_name,
# MAGIC
# MAGIC     MAX(agents_sequential) AS agents_sequential
# MAGIC
# MAGIC FROM call_level
# MAGIC
# MAGIC GROUP BY
# MAGIC     customer_id,
# MAGIC     canonical_ticket_id,
# MAGIC     ticket_dt,
# MAGIC     first_call_ts,
# MAGIC     total_calls,
# MAGIC     agents_involved,
# MAGIC     sticky_agent_name,
# MAGIC     bsd_tier,
# MAGIC     offer_max_loan_amount,
# MAGIC     furthest_stage
# MAGIC
# MAGIC ORDER BY first_call_date_timestamp
# MAGIC
# MAGIC """
# MAGIC
# MAGIC df  = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)

In [ ]:
# Python cell
import os
import pandas as pd

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_urls = [
    "1iWBV4WBiS_Gq1SYIxIzbjXb9bBEy0maZ5mV5I2g-lnY"  # New V4
]

worksheet_name = "Updated_NC_Dump"

combined_df = pd.concat([df], ignore_index=True) 

for sheet_id in gsheet_urls:
    # GoogleSheetReader expects the full URL with gid
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/edit?gid=1325658883#gid=1325658883"
    
    print(f"Writing to {sheet_id}...")
    gs_reader = GoogleSheetReader(url, service_account)
    
    gs_reader.clear_sheets(
        worksheet_name,
        start_row=1,
        start_column=1
    )
    
    gs_reader.set_df_in_sheets( 
        worksheet_name, 
        combined_df, 
        include_headers=True, 
        row=1, 
        col=1
    )
print("Done writing to workbook!")